# Tutorial 52: General Creation of Element Dataframes

This Example demonstrates the capabilities of the class Dataframes_SIR3S_Model that extends SIR3S_Model be abilities to work directley with pandas dataframes. It is shown how to create dataframes containing information about elements such as Nodes, Pipes, etc. existing in a SIR 3S Model. The methods presented are not user-defined and neither efficient, but get you the most important information quickly. For more detailed methods of creating dataframes, see Tutorial 51.

# Toolkit Release

In [1]:
#pip install 

# Imports

## SIR 3S Toolkit

### Regular Import/Init

In [2]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [3]:
from sir3stoolkit.core import wrapper

In [4]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [5]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-11-25 16:23:47,150] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:23:47,150] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:23:47,178] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-11-25 16:23:47,305] INFO in sir3stoolkit.core.wrapper: [Initialization] Sir3S_Toolkit.dll file version 90.15.11.0 (matches this sir3stoolkit version).


### Additional Import/Init for Dataframes class

In [6]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [7]:
s3s = SIR3S_Model_Dataframes()

[2026-11-25 16:23:48,754] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


# Open Model

The tutorial uses *Network 3*, a fictional district heating network shared by the mantle tutorials: a heat plant (in its own container "Heizwerk"), about 180 consumers, a ring road and two longitudinal sections, calculated over one day. It is built from scratch with the SIR 3S Toolkit by [Network3.py](https://github.com/3SConsult/sir3stoolkit/blob/main/docs/source/networks/Network3.py); `Network3.db3` is included in the tutorial's ZIP archive.

Its elements have unique ID references (IDREFERENZ), e.g. `VL-HS-01` for the first supply pipe of the main street. They are used below to look up tks with [GetTkFromIDReference()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetTkFromIDReference) - the tks themselves differ between builds of the model.

In [8]:
s3s.OpenModel(dbName=r"Network3.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

[2026-11-25 16:23:54,531] INFO in sir3stoolkit.core.wrapper: Model is open for further operation


# Calculate Model

In [9]:
s3s.ExecCalculation(True) # To ensure result data

[2026-11-25 16:23:59,623] INFO in sir3stoolkit.core.wrapper: Model Calculation is complete


# Generate Element Dataframes

We can use the [generate_element_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_element_dataframe) method to quickly generate basic dataframes containing all instances of hydraulic element types (Node, Pipe, etc.) in a SIR 3S model. 

All model_data and result values (self.GetResultProperties_from_elementType(onlySelectedVectors=False)) for the static timestamp are included. Result values are given as floats.

The pd.Dataframe will automatically be transformed into a gpd.GeoDataFrame if a SRID is defined in the model, after a geometry column is created.

In [10]:
object_types = [item for item in dir(s3s.ObjectTypes) if not (item.startswith('__') and item.endswith('__'))]
print(object_types) # Check for hydraulic elmement types

['AGSN_HydraulicProfile', 'AirVessel', 'Arrow', 'Atmosphere', 'BlockConnectionNode', 'CalcPari', 'CharacteristicLossTable', 'CharacteristicLossTable_Row', 'Circle', 'Compressor', 'CompressorTable', 'CompressorTable_Row', 'ControlEngineeringNexus', 'ControlMode', 'ControlPointTable', 'ControlPointTable_Row', 'ControlValve', 'ControlVariableConverter', 'ControlVariableConverterRSTE', 'CrossSectionTable', 'CrossSectionTable_Row', 'DPGR_DPKT_DatapointDpgrConnection', 'DPGR_DataPointGroup', 'DPKT_Datapoint', 'DamageRatesTable', 'DamageRatesTable_Row', 'DeadTimeElement', 'Demand', 'DifferentialRegulator', 'DirectionalArrow', 'DistrictHeatingConsumer', 'DistrictHeatingFeeder', 'Divider', 'DriveEfficiencyTable', 'DriveEfficiencyTable_Row', 'DrivePowerTable', 'DrivePowerTable_Row', 'EBES_FeederGroups', 'EfficiencyConverterTable', 'EfficiencyConverterTable_Row', 'ElementQuery', 'EnergyRecoveryTable', 'EnergyRecoveryTable_Row', 'EnvironmentTemp', 'FWBZ_DistrictHeatingReferenceValues', 'FlapValve'

This function allows for little user definition the only paramters are element_type and tks of that element type to exclusively use. For more user defined dataframe creation see Tutorial 51.

## Node

We can parametrize the function to only include certain tks of the element type in the dataframe.

In [11]:
good_tks = []

In [12]:
for tk in s3s.GetTksofElementType(s3s.ObjectTypes.Node):
    if s3s.GetValue(tk, "FkCont")[0] == s3s.GetMainContainer()[0]: # Main container
        good_tks.append(tk)

In [13]:
df_node, timestamp_to_tuple_index = s3s.generate_element_dataframe(element_type=s3s.ObjectTypes.Node, tks=None)
df_node.head(3)

[2026-11-25 16:23:59,692] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df for element type: ObjectTypes.Node ...


[2026-11-25 16:23:59,692] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df_model_data for element type: ObjectTypes.Node ...


[2026-11-25 16:23:59,698] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node


[2026-11-25 16:23:59,699] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 592 element(s) of element type ObjectTypes.Node.


[2026-11-25 16:23:59,710] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.Node.


[2026-11-25 16:23:59,710] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 37 model_data properties.


... (17 outputs left out)

[2026-11-25 16:24:05,389] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.1 s).


[2026-11-25 16:24:05,392] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (1, 15392) (4.7 s in total)


[2026-11-25 16:24:05,400] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Merging df_model_data with df_results for element type: ObjectTypes.Node ...


,tk,Name,Ktyp,Zkor,QmEin,Lfakt,Fkpzon,Fkfstf,Fkutmp,Fkfqps,...,PHMINMAXDIF,PH_EIN,PH_MIN,PMAX_INST,PMIN_INST,QM,RHO,T,TTR,VOLD
0,4611793922327382578,R-H-BWN-04O,QKON,199.73,0,1,-1,5233359644433413774,5363574168628413302,-1,...,"(0.0,)","(4.065771,)","(0.0,)","(5.065771,)","(5.065771,)","(0.0,)","(983.7,)","(60.0,)","(0.0,)","(0.0,)"
1,4614525762964559797,R-H-LSN-06W,QKON,213.43,0,1,-1,5233359644433413774,5363574168628413302,-1,...,"(0.0,)","(2.8121,)","(0.0,)","(3.8121,)","(3.8121,)","(0.0,)","(983.7,)","(60.0,)","(0.0,)","(0.0,)"
2,4615306792408208499,V-HS-12,QKON,210.61,0,1,-1,5233359644433413774,5363574168628413302,-1,...,"(0.0,)","(6.266668,)","(0.0,)","(7.266668,)","(7.266668,)","(0.0,)","(969.2823,)","(84.52936,)","(0.5989156,)","(0.0,)"


All results properties are written as tuples. 

In [14]:
df_node["DP"].iloc[0][0]

3.127841

## Pipe

We can also request a list of timestamps (int or str) to be included in the results.

In [15]:
s3s.GetTimeStamps()[0]

['2026-01-15 00:00:00.000 +01:00',
 '2026-01-15 02:00:00.000 +01:00',
 '2026-01-15 04:00:00.000 +01:00',
 '2026-01-15 06:00:00.000 +01:00',
 '2026-01-15 08:00:00.000 +01:00',
 '2026-01-15 10:00:00.000 +01:00',
 '2026-01-15 12:00:00.000 +01:00',
 '2026-01-15 14:00:00.000 +01:00',
 '2026-01-15 16:00:00.000 +01:00',
 '2026-01-15 18:00:00.000 +01:00',
 '2026-01-15 20:00:00.000 +01:00',
 '2026-01-15 22:00:00.000 +01:00',
 '2026-01-16 00:00:00.000 +01:00']

Let's include the first three timestamps.

In [16]:
df_pipe, timestamp_to_tuple_index = s3s.generate_element_dataframe(element_type=s3s.ObjectTypes.Pipe
                                                                    ,timestamps=[0, 1, 2])
                                

[2026-11-25 16:24:05,598] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df for element type: ObjectTypes.Pipe ...


[2026-11-25 16:24:05,602] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df_model_data for element type: ObjectTypes.Pipe ...


[2026-11-25 16:24:05,602] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe


[2026-11-25 16:24:05,605] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 572 element(s) of element type ObjectTypes.Pipe.


[2026-11-25 16:24:05,605] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.Pipe.


[2026-11-25 16:24:05,608] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 46 model_data properties.


... (17 outputs left out)

[2026-11-25 16:24:24,173] INFO in sir3stoolkit.mantle.dataframes: [results] 30316 fully NaN columns dropped.


[2026-11-25 16:24:24,173] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.1 s).


[2026-11-25 16:24:24,183] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (3, 21880) (16.6 s in total)


[2026-11-25 16:24:24,187] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Merging df_model_data with df_results for element type: ObjectTypes.Pipe ...


In [17]:
df_pipe.head()

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,...,VAV,VI,VK,VOLDA,WVL,ZAUS,ZEIN,ZVEC_end,ZVEC_sequence,ZVEC_start
0,4612274316321154308,Rohr V-KAN-05 V-H-KAN-05O,5741544704281189497,4778210553065428736,-1,15,0,0.1,1,0.025,...,"(0.2322339, 0.2120379, 0.2322075)","(0.2322564, 0.2120621, 0.2322309)","(0.2322096, 0.2120136, 0.2321844)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(208.7, 208.7, 208.7)","((208.31, 208.7), (208.31, 208.7), (208.31, 20...","(208.31, 208.31, 208.31)"
1,4614940901611765838,Rohr V-AWN-04 V-H-AWN-04O,5741544704281189497,4778210553065428736,-1,15,0,0.1,1,0.025,...,"(0.2406659, 0.2194403, 0.2406915)","(0.2406869, 0.219462, 0.2407133)","(0.2406435, 0.2194185, 0.2406698)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(197.97, 197.97, 197.97)","((198.04, 197.97), (198.04, 197.97), (198.04, ...","(198.04, 198.04, 198.04)"
2,4618102496224966047,Rohr V-AWS-03 V-H-AWS-03O,5741544704281189497,4778210553065428736,-1,15,0,0.1,1,0.025,...,"(0.1050423, 0.09591155, 0.1050787)","(0.1050637, 0.09593386, 0.1051005)","(0.1050199, 0.09588925, 0.1050568)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(200.24, 200.24, 200.24)","((200.32, 200.24), (200.32, 200.24), (200.32, ...","(200.32, 200.32, 200.32)"
3,4618494560130677338,Rohr R-BWS-01 R-BWS-A1,5727924218879267289,4778210553065428736,-1,50,0,0.1,1,0.025,...,"(0.6121801, 0.5711334, 0.6351361)","(0.6121851, 0.5711401, 0.6351428)","(0.6121717, 0.5711267, 0.6351294)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(199.02, 199.02, 199.02)","((199.72, 199.02), (199.72, 199.02), (199.72, ...","(199.72, 199.72, 199.72)"
4,4619202338427165336,Rohr V-AWN-06 V-AWN-07,5424610760398623711,4778210553065428736,-1,50,0,0.1,1,0.025,...,"(0.3633023, 0.336959, 0.3700724)","(0.3633249, 0.3369825, 0.370096)","(0.363278, 0.3369356, 0.3700493)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(0.0, 0.0, 0.0)","(201.1, 201.1, 201.1)","((199.92, 201.1), (199.92, 201.1), (199.92, 20...","(199.92, 199.92, 199.92)"


Each result property is written as tuple, where each entry corresponds to one timestamp.

In [18]:
df_pipe["VAV"].iloc[0]

(0.2322339, 0.2120379, 0.2322075)

We can use the returned mapping for easy access via literal timestamp.

In [19]:
timestamp_to_tuple_index

{'2026-01-15 00:00:00.000 +01:00': 0,
 '2026-01-15 02:00:00.000 +01:00': 1,
 '2026-01-15 04:00:00.000 +01:00': 2}

In [20]:
df_pipe["VAV"].iloc[0][timestamp_to_tuple_index["2026-01-15 02:00:00.000 +01:00"]]

0.2120379

For already vectorized data (pipe interior points) we have a tuple of tuples with the time tuple as the higher level one.

In [21]:
df_pipe["TTRVEC_sequence"].iloc[0]

((0.5195163, 0.5374579), (0.5601741, 0.5798247), (0.5077574, 0.5257011))

In [22]:
df_pipe["TTRVEC_sequence"].iloc[0][timestamp_to_tuple_index["2026-01-15 02:00:00.000 +01:00"]]

(0.5601741, 0.5798247)

## DistrictHeatingConsumer

We can specify the timestamp we want the result data for either as "2026-01-15 02:00:00.000 +01:00" or 1 meaning the second element in s3s.GetTimeStamps()[0], otherwise static is used.

In [23]:
df_dhc, timestamp_to_tuple_index = s3s.generate_element_dataframe(element_type=s3s.ObjectTypes.DistrictHeatingConsumer, tks=None, timestamps=[1])
df_dhc.head(3)

[2026-11-25 16:24:24,429] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df for element type: ObjectTypes.DistrictHeatingConsumer ...


[2026-11-25 16:24:24,433] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Generating df_model_data for element type: ObjectTypes.DistrictHeatingConsumer ...


[2026-11-25 16:24:24,433] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.DistrictHeatingConsumer


[2026-11-25 16:24:24,434] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 175 element(s) of element type ObjectTypes.DistrictHeatingConsumer.


[2026-11-25 16:24:24,435] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.DistrictHeatingConsumer.


[2026-11-25 16:24:24,436] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 55 model_data properties.


... (18 outputs left out)

[2026-11-25 16:24:25,903] INFO in sir3stoolkit.mantle.dataframes: [results] DataFrame assembled (0.0 s).


[2026-11-25 16:24:25,905] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (1, 3150) (0.8 s in total)


[2026-11-25 16:24:25,905] INFO in sir3stoolkit.mantle.dataframes: [generate_element_dataframe] Merging df_model_data with df_results for element type: ObjectTypes.DistrictHeatingConsumer ...


,tk,Name,Beschreibung,Ind0,W0,Qm0,Tvl0,Trs0,Lfk,Rho0,...,PHIRL,PHIVL,QM,TI,TK,TVMIN,W,WHYUV,WSOLL,WTHUV
0,4619274837930594515,Fernwärmeverbraucher V-H-AWN-06O R-H-AWN-06O,Wohnhaus Ahornweg Nord 12,0,22.8,1,90,60,1,1000,...,"(-3.333333e+32,)","(-3.333333e+32,)","(0.4119196,)","(83.78574,)","(60.0,)","(75.01807,)","(11.40001,)","(-1.430511e-05,)","(11.4,)","(0.0,)"
1,4626565215742718797,Fernwärmeverbraucher V-H-AWN-02O R-H-AWN-02O,Wohnhaus Ahornweg Nord 4,0,25.9,1,90,60,1,1000,...,"(-3.333333e+32,)","(-3.333333e+32,)","(0.4570484,)","(84.35065,)","(60.0,)","(75.01807,)","(12.95,)","(0.0,)","(12.95,)","(0.0,)"
2,4628630515449372392,Fernwärmeverbraucher V-H-AWN-05W R-H-AWN-05W,Wohnhaus Ahornweg Nord 9,0,18.0,1,90,60,1,1000,...,"(-3.333333e+32,)","(-3.333333e+32,)","(0.3245689,)","(83.83185,)","(60.0,)","(75.01807,)","(9.000011,)","(-1.144409e-05,)","(9.0,)","(0.0,)"
